# Weighted linear combination: build a suitability surface

A weighted linear combination (WLC) combines standardised criterion layers into a composite score. This lab focuses on the aggregation step: criteria are already expressed on a common preference scale, weights express relative importance, and a Boolean mask represents hard exclusions.

The data are synthetic. They are designed to make the arithmetic and modelling assumptions visible, not to represent a real place or support an actual siting decision.

In JupyterLab, run from the repository root. In Colab, the setup cell clones the continuation branch when needed.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import FloatSlider, interact

root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'implementations/python/weighted-overlay').is_dir()), None)
if root is None:
    clone_dir = Path('/tmp/spatial-analysis-compendium')
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch',
        'compendium/section-01-geometry-continuation',
        'https://github.com/geo-dev-hub/spatial-analysis.git', str(clone_dir)
    ], check=True)
    root = clone_dir
sys.path.insert(0, str(root / 'implementations/python/weighted-overlay'))
from weighted_overlay import weighted_linear_combination

print('Repository:', root)

## 1. Create the criterion layers

Each layer has values from 0 to 1, where a larger number means more preferred. In this synthetic example, the layers represent road accessibility, terrain suitability and low flood exposure.

These are already-standardised scores. The notebook does not infer the value functions from raw distances, slope measurements or flood observations.

In [ ]:
rows, cols = 6, 8
y, x = np.mgrid[0:rows, 0:cols]

criteria = {
    'road_access': 1.0 - x / (cols - 1),
    'terrain': 1.0 - np.abs(y - 2.5) / 4.0,
    'low_flood_exposure': 1.0 - np.clip((x + y) / 13.0, 0.0, 1.0),
}
eligible = np.ones((rows, cols), dtype=bool)
eligible[2:4, 3:5] = False  # a synthetic hard-exclusion area

for name, layer in criteria.items():
    assert layer.shape == (rows, cols)
    assert np.all((layer >= 0) & (layer <= 1))
    print(f'{name:22} min={layer.min():.2f} max={layer.max():.2f}')
print('Eligible cells:', int(eligible.sum()), 'of', eligible.size)

## 2. Inspect each factor before combining it

A composite map can hide the behaviour of an individual criterion. Look at the inputs separately before looking at the result. A criterion that has the wrong direction or an inappropriate standardisation function can dominate the output for the wrong reason.

In [ ]:
for name, layer in criteria.items():
    fig, ax = plt.subplots(figsize=(7, 4))
    image = ax.imshow(layer, vmin=0, vmax=1)
    ax.set_title(f'{name.replace("_", " ").title()} — standardised preference')
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    fig.colorbar(image, ax=ax, label='Preference score (0–1)')
    plt.show()

## 3. Choose weights and calculate the composite score

The weights are relative importance values. The implementation normalises them to sum to one. WLC is compensatory: a high score on one factor can offset a lower score on another. The exclusion mask is applied separately so that an excluded cell cannot become eligible through a weighted sum.

In [ ]:
weights = {'road_access': 0.50, 'terrain': 0.30, 'low_flood_exposure': 0.20}
scores = weighted_linear_combination(criteria, weights, eligible=eligible)
print('Weights:', weights)
print('Weight total:', sum(weights.values()))
print('Best eligible cell (row, column):', np.unravel_index(np.nanargmax(scores), scores.shape))
print('Best eligible score:', float(np.nanmax(scores)))

fig, ax = plt.subplots(figsize=(7, 4))
image = ax.imshow(scores, vmin=0, vmax=1)
ax.set_title('Weighted suitability score')
ax.set_xlabel('Column')
ax.set_ylabel('Row')
fig.colorbar(image, ax=ax, label='Composite score')
plt.show()

## 4. Change the weights

Move the controls to see how the composite surface changes. The weights are renormalised by the implementation, so their absolute values matter only through their relative proportions.

If all weights are zero, no weighted average exists; the lab reports that instead of inventing one.

In [ ]:
def explore_weights(road_access=0.50, terrain=0.30, low_flood_exposure=0.20):
    trial = {
        'road_access': road_access,
        'terrain': terrain,
        'low_flood_exposure': low_flood_exposure,
    }
    if sum(trial.values()) == 0:
        print('At least one criterion weight must be positive.')
        return
    result = weighted_linear_combination(criteria, trial, eligible=eligible)
    fig, ax = plt.subplots(figsize=(7, 4))
    image = ax.imshow(result, vmin=0, vmax=1)
    ax.set_title('Suitability under the selected relative weights')
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    fig.colorbar(image, ax=ax, label='Composite score')
    plt.show()
    print('Raw weights:', trial)
    print('Normalised weights:', {k: v / sum(trial.values()) for k, v in trial.items()})
    print('Best eligible cell:', np.unravel_index(np.nanargmax(result), result.shape))

interact(
    explore_weights,
    road_access=FloatSlider(value=0.50, min=0, max=1, step=0.05),
    terrain=FloatSlider(value=0.30, min=0, max=1, step=0.05),
    low_flood_exposure=FloatSlider(value=0.20, min=0, max=1, step=0.05),
)

## 5. Compare scenarios and inspect NoData

A single weight set is one scenario, not the answer. Compare at least one plausible alternative. Also notice that the implementation uses a strict common-footprint rule: if any factor is missing at a cell, that cell is not scored.

In [ ]:
alternative_weights = {'road_access': 0.25, 'terrain': 0.45, 'low_flood_exposure': 0.30}
alternative_scores = weighted_linear_combination(criteria, alternative_weights, eligible=eligible)
valid = np.isfinite(scores) & np.isfinite(alternative_scores)
print('Maximum absolute score change:', float(np.max(np.abs(scores[valid] - alternative_scores[valid]))))
print('Baseline best cell:', np.unravel_index(np.nanargmax(scores), scores.shape))
print('Alternative best cell:', np.unravel_index(np.nanargmax(alternative_scores), alternative_scores.shape))

with_missing = {name: layer.copy() for name, layer in criteria.items()}
with_missing['terrain'][0, 0] = np.nan
missing_scores = weighted_linear_combination(with_missing, weights, eligible=eligible)
print('Cell with a missing criterion:', missing_scores[0, 0])
assert np.isnan(missing_scores[0, 0])

## Questions to take further

1. Which factor has the greatest influence on the result, and how did you decide?
2. What happens if road accessibility is standardised in the opposite direction?
3. Which criteria may measure overlapping aspects of the same underlying phenomenon?
4. Which locations should be excluded by a hard constraint rather than given a low score?
5. How would you justify the weights with affected communities and domain experts?
6. What independent evidence could validate the resulting suitability surface?

## Continue learning

- [Weighted linear combination method](../methods/weighted-linear-combination.md)
- [Python implementation](../implementations/python/weighted-overlay/README.md)
- [Spatial decision-support learning path](../learning-paths/spatial-decision-support.md)

**Remember:** the calculation is straightforward. The criteria, standardisation, weights, constraints and validation determine whether the decision model is defensible.